In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
ProjDIR = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/" # Change to your project directory
sys.path.insert(1, f'{ProjDIR}/src/')
sys.path.insert(1, '/home/jw3514/Work/UNIMED/src')
from CellType_PSY import *
from UNIMED import *
#import scanpy as sc
HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

try:
    os.chdir(f"{ProjDIR}/notebooks/")
    print(f"Current working directory: {os.getcwd()}")
except FileNotFoundError as e:
    print(f"Error: Could not change directory - {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

### Read Matrices and annotation

In [ ]:
HumanCT_Z2_HCT = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/HumanCT.Subcluster.TPM.0.1.Filt.Spec.clip.lowexp.cut1e4.csv", index_col=0)
HumanCT_Z2_HCT.shape

In [ ]:
Subcluster_anno = pd.read_excel("/home/jw3514/Work/CellType_Psy/dat/subcluster_annotation.xlsx",
                               index_col="Subcluster")

In [ ]:
Subcluster_anno.head(3)

In [ ]:
Subcluster_anno.loc[1971, :]

In [ ]:
def AnnotateSubcluster(df, Anno=Subcluster_anno):
    for ID, row in df.iterrows():
        id_ = int(ID.split("-")[0])
        df.loc[ID, "Supercluster"] = Anno.loc[id_, "Supercluster"]
        df.loc[ID, "Cluster"] = Anno.loc[id_, "Cluster"]
        df.loc[ID, "Top ROIGroupFine"] = Anno.loc[id_, "Top ROIGroupFine"]
        df.loc[ID, "Top ROI"] = Anno.loc[id_, "Top ROI"]
        df.loc[ID, "Number of cells"] = Anno.loc[id_, "Number of cells"] #Top three dissections
    return df

In [ ]:
GeneWeightDIR = "../dat/GeneWeights/"
Bias_Save_Dir = "/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/dat/Subcluster_Spec_Bias/"
if not os.path.exists(Bias_Save_Dir): # make dir if not exists
    os.makedirs(Bias_Save_Dir)

In [ ]:
HIQ_GW = Fil2Dict("{}/HIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
LIQ_GW = Fil2Dict("{}/LIQ.top61.nopLI.LGD_Dmis_SameWeight.gw".format(GeneWeightDIR))
print(len(HIQ_GW), len(LIQ_GW))

In [ ]:
HIQ_Subcluster_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, HIQ_GW)
HIQ_Subcluster_Bias = AnnotateSubcluster(HIQ_Subcluster_Bias)
#HIQ_Z2_Bias.to_csv("{}/HCT.ASD61.HIQ.csv".format(Bias_Save_Dir))

LIQ_Subcluster_Bias = HumanCT_AvgZ_Weighted(HumanCT_Z2_HCT, LIQ_GW)
LIQ_Subcluster_Bias = AnnotateSubcluster(LIQ_Subcluster_Bias)
#LIQ_Z2_Bias.to_csv("{}/HCT.ASD61.LIQ.csv".format(Bias_Save_Dir))

In [ ]:
LIQ_Subcluster_Bias.head(20)

In [ ]:
def plot_supercluster_bias(df, effect_col="EFFECT", supercluster_col="Supercluster", figsize=(12,6), title="Distribution of Bias (EFFECT) by Supercluster"):
    """
    Plots the distribution of bias (EFFECT) by supercluster as a boxplot.

    Parameters:
        df (pd.DataFrame): DataFrame containing the data.
        effect_col (str): Name of the column containing the effect/bias values.
        supercluster_col (str): Name of the column containing the supercluster labels.
        figsize (tuple): Figure size for the plot.
        title (str): Title for the plot.
    """
    # Sort Superclusters by median EFFECT
    supercluster_order = (
        df.groupby(supercluster_col)[effect_col]
        .median()
        .sort_values(ascending=False)
        .index
    )

    plt.figure(figsize=figsize)
    sns.boxplot(
        data=df,
        x=supercluster_col,
        y=effect_col,
        order=supercluster_order
    )
    plt.xticks(rotation=45, ha='right')
    plt.title(title)
    plt.ylabel(f"Bias ({effect_col})")
    plt.xlabel(supercluster_col)
    plt.tight_layout()
    plt.show()

# Example usage:
# plot_supercluster_bias(LIQ_Subcluster_Bias)

In [ ]:
plot_supercluster_bias(HIQ_Subcluster_Bias, title="HIQ")
plot_supercluster_bias(LIQ_Subcluster_Bias, title="LIQ")

# PBS with subcluster

In [ ]:
Mut_n_IQ_conf = pd.read_csv("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/dat/Pheno_Bias_vs_IQ/Mut_n_IQ_conf.csv", index_col=0)
Avg_Gene_IQ_DF = pd.read_csv("/home/jw3514/Work/CellType_Psy/CellTypeBias_VIP/dat/Pheno_Bias_vs_IQ/Mut_n_IQ_conf.GeneL.csv", index_col=0)
Mut_n_IQ_conf.head(2)
Avg_Gene_IQ_DF.head(2)

In [ ]:
def linear_fit(biases, IQs, alpha=0.05):
    model = sm.OLS(IQs, sm.add_constant(biases))
    results = model.fit()
    
    intercept = results.params[0]
    beta = results.params[1]
    # get CI of beta
    ci = results.conf_int(alpha=alpha)

    ci_low = ci[1][0]
    ci_high = ci[1][1]     
    r_value = results.rsquared
    p_value = results.pvalues[1]
    std_err = results.bse[1]
    rho, p_rho = spearmanr(biases, IQs)
    r, p_r = pearsonr(biases, IQs)
    
    return intercept, beta, ci_low, ci_high, r_value, p_value, std_err, rho, p_rho, r, p_r

def Make_HumanCT_DF(Mut_n_IQ_conf, HCT_Z2_MAT_HCT, Anno, output_file, alpha=0.05):
    names, cluster, supercluster, spearmanr, spearmanp, pearsonr, pearsonp, beta_values, beta_ci_low, beta_ci_high, intercept_values, r_value_values, p_value_values, std_err_values = [], [],[],[],[],[],[],[],[],[],[],[],[],[]
    for i, Idx in enumerate(HCT_Z2_MAT_HCT.columns.values):
        biases, IQs = BiasVsPheno(Mut_n_IQ_conf, HCT_Z2_MAT_HCT , Idx, 'xx')
        intercept, beta, ci_low, ci_high, r_value, p_value, std_err, rho, p_rho, r, p_r = linear_fit(biases, IQs, alpha=0.05)
        
        names.append("{}".format(Idx))

        cluster.append(Anno.loc[i, "Cluster"])
        supercluster.append(Anno.loc[i, "Supercluster"])
        spearmanr.append(rho)
        spearmanp.append(p_rho)
        pearsonr.append(r)
        pearsonp.append(p_r)
        beta_values.append(beta)
        beta_ci_low.append(ci_low)
        beta_ci_high.append(ci_high)
        intercept_values.append(intercept)
        r_value_values.append(r_value)
        p_value_values.append(p_value)
        std_err_values.append(std_err)

    str_res_df = pd.DataFrame(data={"CT":names, "Cluster":cluster, "Supercluster":supercluster, "SpearmanR":spearmanr, "SpearmanP":spearmanp, 
                                            "PearsonR":pearsonr, "PearsonP":pearsonp, "beta":beta_values, "CI_low":beta_ci_low, "CI_high":beta_ci_high, "intercept":intercept_values, "r_value":r_value_values, 
                                            "p_value":p_value_values, "std_err":std_err_values})
    str_res_df = str_res_df.sort_values("SpearmanR")
    #str_res_df = ADJ_P(str_res_df)
    str_res_df.to_csv(output_file)
    return str_res_df

In [ ]:
HumanCT_res_df_GeneL = Make_HumanCT_DF(Avg_Gene_IQ_DF, HumanCT_Z2_HCT, Subcluster_anno, "../dat/Pheno_Bias_vs_IQ/HumanCT.Subclusterspec.GeneL.csv")

In [ ]:
HumanCT_res_df_GeneL = 

In [ ]:
HumanCT_res_df_GeneL.head(20)

In [ ]:
SuperClusterBias_BoxPlot_CorrIQ(HumanCT_res_df_GeneL, flip_axis=True, figsize=(6, 8), plot_metric="beta")

In [ ]:
TPM = pd.read_csv("/home/jw3514/Work/CellType_Psy/dat/HumanCTExpressionMats/HumanCT.Subcluster.TPM.0.1.Filt.csv", index_col=0)

In [ ]:
TPM.head(2)

In [ ]:
HumanCT_res_df_GeneL.head(2)

In [ ]:
PBS_CGE = HumanCT_res_df_GeneL[HumanCT_res_df_GeneL["Supercluster"]=="CGE interneuron"].copy()
PBS_CGE = PBS_CGE.set_index("CT")
PBS_CGE["VIP_ExpL"] = TPM.loc[GeneSymbol2Entrez["VIP"], PBS_CGE.index]

In [ ]:
PBS_CGE

In [ ]:
#PBS_CGE["VIP_ExpL"].hist()
plt.hist(np.log10(PBS_CGE["VIP_ExpL"]))

In [ ]:
VIP_Cutoff = 100
VIP_Pos = PBS_CGE[PBS_CGE["VIP_ExpL"] > VIP_Cutoff]
VIP_Neg = PBS_CGE[PBS_CGE["VIP_ExpL"] < VIP_Cutoff]

In [ ]:
data = [VIP_Pos["beta"], VIP_Neg["beta"]]
# Perform Mann-Whitney U test
stat, pval = scipy.stats.mannwhitneyu(VIP_Pos["beta"], 
                                    VIP_Neg["beta"], alternative="less")
# Create boxplot with individual points
bp = plt.boxplot(data, labels=["VIP+", "VIP-"])
# Add scatter points
for i, d in enumerate([VIP_Pos["beta"], VIP_Neg["beta"]]):
    x = np.random.normal(i+1, 0.04, size=len(d))
    plt.scatter(x, d, alpha=0.4, s=20)
plt.ylabel("Effect")
plt.title(f"p = {pval:.2e}")
plt.show()